# 00 Colab Setup

Purpose: mount Google Drive, verify the mirrored Drive source folder, confirm the dataset-build artifacts are present, install pinned runtime dependencies from that source, and resolve the persistent runtime paths.

Final workflow:
1. Author and validate code locally in the Git repo.
2. Build the dataset manifests locally with `./.venv/bin/python scripts/build_dataset_manifests.py --profile full`.
3. Push the execution subset into Google Drive from the local machine with `make drive-push-source`.
4. Mount Drive in Colab.
5. Run this notebook.
6. Execute the stage notebook you need from the Drive-backed source tree.

Persistence model:
- `/content/drive/MyDrive/json-ft-source` is the mirrored execution copy of the repo.
- `/content/drive/MyDrive/json-ft-runs` stores runtime outputs and survives Colab restarts.
- Source refresh happens on the local machine through explicit Drive sync commands, not through Colab upload staging.


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

SOURCE_ROOT = Path('/content/drive/MyDrive/json-ft-source')
RUNTIME_ROOT = Path('/content/drive/MyDrive/json-ft-runs')

required_paths = [
    SOURCE_ROOT / 'src',
    SOURCE_ROOT / 'scripts',
    SOURCE_ROOT / 'configs',
    SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_canonical.jsonl',
    SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_eval_manifest.jsonl',
    SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_sft_messages.jsonl',
    SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_dataset_build_summary.json',
    SOURCE_ROOT / 'requirements-colab.txt',
]
missing = [path for path in required_paths if not path.exists()]

print(f'Source root: {SOURCE_ROOT}')
print(f'Runtime root: {RUNTIME_ROOT}')
if missing:
    missing_text = '\n'.join(str(path) for path in missing)
    raise FileNotFoundError(
        'Drive source is incomplete. Run make drive-push-source locally before continuing:\n' + missing_text
    )
print('Drive source preflight passed.')


Source root: /content/drive/MyDrive/json-ft-source
Runtime root: /content/drive/MyDrive/json-ft-runs
Drive source preflight passed.


In [3]:
!python -m pip install --upgrade pip
!python -m pip install -r /content/drive/MyDrive/json-ft-source/requirements-colab.txt


In [4]:
import sys

sys.path.insert(0, str(SOURCE_ROOT / 'src'))

from json_ft.runtime import resolve_runtime_context, format_runtime_summary

context = resolve_runtime_context(
    repo_root=SOURCE_ROOT,
    stage='setup',
    run_name='colab-session',
    runtime_root=RUNTIME_ROOT,
)
print(format_runtime_summary(context))
print('\nColab Drive-source setup is ready.')


is_colab=True
repo_root=/content/drive/MyDrive/json-ft-source
runtime_root=/content/drive/MyDrive/json-ft-runs
persistent_root=/content/drive/MyDrive/json-ft-runs/persistent
scratch_root=/content/drive/MyDrive/json-ft-runs/scratch
plots_dir=/content/drive/MyDrive/json-ft-runs/persistent/plots
stage=setup
run_name=colab-session
run_dir=/content/drive/MyDrive/json-ft-runs/persistent/setup/colab-session

Colab Drive-source setup is ready.
